# KNOWLY 分析ノートブック①　架空データで「出題方式」を比べる

**このノートブックでやること**

1. 生徒20人・理科50問の「架空の解答データ」を作る(忘れていく様子を、仮のルールで再現します)
2. 2つの出題方式で、30日間の学習をシミュレーションする
   - **ランダム**:毎日、問題をランダムに選ぶ
   - **SM-2**:間隔反復の定番の方式。忘れそうな問題を優先して出す
3. 学習をやめて2週間あけたあと、どちらがよく覚えているかを比べる

**注意**:ここで使うのは本物のデータではなく、仮のルールで作った架空のデータです。
結果は「仕組みが動くことの確認」であり、「効果の証明」ではありません。
本物の効果は、実際に使ってもらったデータで確かめます。

**使い方**:上のメニューから「すべて実行」を押すと、上から順に実行されます。

## 1. 準備と設定

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from pathlib import Path

# グラフで日本語を表示するための設定(Windowsに入っているフォントを順に探す)
plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "MS Gothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# ---- 設定(ここを変えると結果が変わります) ----
N_STUDENTS = 20      # 生徒の人数
N_QUESTIONS = 50     # 問題の数(理科)
STUDY_DAYS = 30      # 学習する日数
PER_DAY = 10         # 1日に出す問題数
REST_DAYS = 14       # 学習をやめてから、最終テストまであける日数
SEED = 42            # 乱数の種(同じ値なら、毎回同じ結果になります)
START_DATE = datetime(2026, 11, 1, 18, 0)   # 学習を始める日時(架空)

DATA_DIR = Path("data")      # 作ったデータの保存先
DATA_DIR.mkdir(exist_ok=True)

## 2. 忘れていく様子を、仮のルールで作る

人は時間がたつと忘れます。そのようすを、よく使われる「忘却曲線」の式で表します。

> 覚えている確率 = 勘で当たる確率 + (1 − 勘で当たる確率) × exp( −経過日数 ÷ S )

忘れてしまっても、勘で当たることはあるので、確率はゼロまで下がらず「勘で当たる確率」(ここでは20%)で止まります。

**S は「覚えの強さ」(日数)** です。S が大きいほど、忘れにくくなります。

- 初めて見たとき、S は生徒の得意さと問題の難しさで決まります。
- 思い出せたときは、S が大きくなります。**忘れかけたときに思い出せたほど、大きく伸びます**(間隔をあける効果)。
- 思い出せなかったときは、S が小さくなります。

この数字(1.5 や 2.0 など)は、あくまで仮の値です。

In [ ]:
rng_master = np.random.default_rng(SEED)
ability = rng_master.normal(0, 1, N_STUDENTS)       # 生徒の得意さ(大きいほど得意)
difficulty = rng_master.normal(0, 1, N_QUESTIONS)   # 問題の難しさ(大きいほど難しい)

# 初めて見たときの「覚えの強さ」(日数)。形は(生徒の数, 問題の数)
S0 = 1.5 * np.exp(0.5 * ability[:, None] - 0.5 * difficulty[None, :])

PRIOR = 0.2   # 勘や知識で当てられる確率(一度も見ていない問題も、忘れた問題もこの値)


def recall_prob(S, elapsed_days):
    """経過日数から、いまその問題に正解できる確率を返す(忘却曲線)。勘で当たる分(PRIOR)が下限"""
    return PRIOR + (1 - PRIOR) * np.exp(-elapsed_days / S)


def new_state():
    """記憶の状態を作る。S=覚えの強さ、last=最後に解いた日(nan=まだ解いていない)"""
    return {
        "S": S0.copy(),
        "last": np.full((N_STUDENTS, N_QUESTIONS), np.nan),
    }


def answer(state, i, q, day, rng):
    """生徒 i が問題 q に day 日目に答える。(正誤, 解答秒数)を返し、記憶の状態も更新する"""
    last = state["last"][i, q]
    if np.isnan(last):
        p = PRIOR                                         # 初見
    else:
        p = recall_prob(state["S"][i, q], day - last)     # 覚えている確率
    correct = bool(rng.random() < p)

    # 覚えているほど早く答えられる(ばらつきも足す)
    sec = float(np.clip(rng.normal(14 - 9 * p, 2.5), 2, 40))

    # 解答のあとで、記憶の強さを更新する
    if np.isnan(last):
        pass                                              # 初見は、答えを見ただけ(S0 のまま)
    elif correct:
        state["S"][i, q] *= 1.2 + 2.0 * (1 - p)           # 忘れかけで思い出せたほど強く定着
    else:
        state["S"][i, q] = max(0.5 * S0[i, q], 0.4 * state["S"][i, q])
    state["last"][i, q] = day
    return correct, sec


def mean_recall(state, day):
    """day 日目の朝の時点で、全員・全問題の「正解できる確率」の平均と、生徒ごとの平均を返す"""
    elapsed = day - state["last"]
    R = np.where(np.isnan(state["last"]), PRIOR, recall_prob(state["S"], elapsed))
    return R.mean(), R.mean(axis=1)

## 3. 2つの出題方式

**ランダム**:毎日、問題をランダムに10問選びます。

**SM-2(簡易版)**:問題ごとに「次に出す日」を決めておき、期限が来た問題を優先して出します。

- 間違えたら、次の日にまた出します。
- 正解したら、間隔を 1日 → 6日 → (前の間隔 × 難易度の係数)と広げていきます。
- 本来の SM-2 は6段階で自己採点しますが、KNOWLY は「正解/もう一度」の2択です。
  そこで、**正解でも、すぐ答えられたか(8秒未満)で段階を分けます**。ここで解答秒数を使っています。

In [ ]:
def pick_random(state, sched, i, day, rng):
    """ランダム方式:問題をランダムに選ぶ"""
    return [int(q) for q in rng.choice(N_QUESTIONS, PER_DAY, replace=False)]


def pick_sm2(state, sched, i, day, rng):
    """SM-2方式:期限が来た問題を、期限が古い順に出す。足りなければ未学習の問題で埋める"""
    due = sched["due"][i]
    seen = ~np.isnan(state["last"][i])

    chosen = [int(q) for q in np.argsort(due) if seen[q] and due[q] <= day][:PER_DAY]

    if len(chosen) < PER_DAY:      # 足りない分は、まだ解いていない問題から
        unseen = [int(q) for q in rng.permutation(N_QUESTIONS) if not seen[q]]
        chosen += unseen[:PER_DAY - len(chosen)]

    if len(chosen) < PER_DAY:      # それでも足りなければ、期限が近い順に
        rest = [int(q) for q in np.argsort(due) if int(q) not in chosen]
        chosen += rest[:PER_DAY - len(chosen)]
    return chosen


def new_schedule():
    """SM-2 の予定表。due=次に出す日、reps=連続正解の回数、ease=易しさ、interval=間隔(日)"""
    shape = (N_STUDENTS, N_QUESTIONS)
    return {
        "due": np.full(shape, np.inf),
        "reps": np.zeros(shape, dtype=int),
        "ease": np.full(shape, 2.5),
        "interval": np.zeros(shape),
    }


def sm2_update(sched, i, q, day, correct, sec):
    """解答の結果から、次に出す日を決める(SM-2 の簡易版)"""
    if not correct:
        sched["reps"][i, q] = 0
        sched["interval"][i, q] = 1
    else:
        quality = 5 if sec < 8 else 4          # すぐ答えられたら5、時間がかかったら4
        sched["reps"][i, q] += 1
        if sched["reps"][i, q] == 1:
            sched["interval"][i, q] = 1
        elif sched["reps"][i, q] == 2:
            sched["interval"][i, q] = 6
        else:
            sched["interval"][i, q] = round(sched["interval"][i, q] * sched["ease"][i, q])
        e = sched["ease"][i, q] + 0.1 - (5 - quality) * (0.08 + (5 - quality) * 0.02)
        sched["ease"][i, q] = max(1.3, e)
    sched["due"][i, q] = day + sched["interval"][i, q]

## 4. シミュレーションを動かす

30日間の学習のあと、14日あけて、最終日(44日目)の「正解できる確率」を測ります。
解答の記録は、設計メモで決めた6項目(生徒ID、問題ID、解答日時、正誤、解答秒数、挑戦回数)の形で残します。

In [ ]:
LOG_COLUMNS = ["student_id", "question_id", "answered_at", "correct", "response_sec", "attempt_no"]
TEST_DAY = STUDY_DAYS + REST_DAYS


def run_policy(picker, use_sm2, seed):
    """1つの出題方式で、学習から最終テストまでを動かす"""
    rng = np.random.default_rng(seed)
    state = new_state()
    sched = new_schedule()
    attempts = np.zeros((N_STUDENTS, N_QUESTIONS), dtype=int)
    rows = []
    daily = []                          # 毎朝の「正解できる確率」の平均

    for day in range(TEST_DAY + 1):
        mean_all, mean_by_student = mean_recall(state, day)
        daily.append(mean_all)
        if day >= STUDY_DAYS:           # 学習をやめたあとは、測るだけ
            continue
        for i in range(N_STUDENTS):
            offset = 0.0                # その日の学習を始めてからの経過秒数
            for q in picker(state, sched, i, day, rng):
                correct, sec = answer(state, i, q, day, rng)
                attempts[i, q] += 1
                rows.append((
                    f"S{i + 1:03d}",
                    f"Q{q + 1:03d}",
                    START_DATE + timedelta(days=day, minutes=i * 5, seconds=round(offset)),
                    int(correct),
                    round(sec, 1),
                    int(attempts[i, q]),
                ))
                offset += sec + 3       # 解答時間 + 次の問題へ進む時間
                if use_sm2:
                    sm2_update(sched, i, q, day, correct, sec)

    log = pd.DataFrame(rows, columns=LOG_COLUMNS)
    final_all, final_by_student = mean_recall(state, TEST_DAY)
    return log, np.array(daily), final_by_student


log_random, daily_random, final_random = run_policy(pick_random, False, SEED)
log_sm2, daily_sm2, final_sm2 = run_policy(pick_sm2, True, SEED)

# 記録をCSVで保存する(Excelで開いても文字化けしないようにする)
log_random.to_csv(DATA_DIR / "log_random.csv", index=False, encoding="utf-8-sig")
log_sm2.to_csv(DATA_DIR / "log_sm2.csv", index=False, encoding="utf-8-sig")

print("ランダム方式の記録:", len(log_random), "行")
print("SM-2方式の記録  :", len(log_sm2), "行")
log_sm2.head(10)

## 5. 結果を比べる

左のグラフは、毎朝の「正解できる確率」の平均です。右のグラフは、最終日(学習をやめて2週間後)の結果です。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# 左:覚えている確率の推移
ax = axes[0]
days = np.arange(TEST_DAY + 1)
ax.plot(days, daily_random * 100, label="ランダム", color="tab:gray")
ax.plot(days, daily_sm2 * 100, label="SM-2", color="tab:blue")
ax.axvline(STUDY_DAYS, color="black", linestyle="--", linewidth=0.8)
ax.text(STUDY_DAYS + 0.5, 0.03, "学習をやめる", fontsize=9, transform=ax.get_xaxis_transform())
ax.set_xlabel("日数")
ax.set_ylabel("正解できる確率の平均(%)")
ax.set_title("正解できる確率の推移")
ax.legend()

# 右:最終日の結果(生徒ごとのばらつきも点で表示)
ax = axes[1]
means = [final_random.mean() * 100, final_sm2.mean() * 100]
ax.bar(["ランダム", "SM-2"], means, color=["tab:gray", "tab:blue"], alpha=0.6)
for x, vals in enumerate([final_random, final_sm2]):
    ax.scatter(np.full(len(vals), x) + np.random.default_rng(0).uniform(-0.15, 0.15, len(vals)),
               vals * 100, color="black", s=12)
ax.set_ylabel("正解できる確率(%)")
ax.set_title(f"最終日({TEST_DAY}日目)の結果(点=生徒1人)")
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

print(f"最終日の平均  ランダム: {final_random.mean() * 100:.1f}%   SM-2: {final_sm2.mean() * 100:.1f}%")
better = int((final_sm2 > final_random).sum())
print(f"SM-2 の方が高かった生徒: {N_STUDENTS}人中 {better}人")

## 6. たまたまではないか確認する(くり返し)

1回の結果は、乱数の偶然かもしれません。乱数の種を変えて20回くり返し、結果が安定しているかを見ます。

In [ ]:
REPEATS = 20
rand_scores, sm2_scores = [], []
for seed in range(REPEATS):
    _, _, f_r = run_policy(pick_random, False, seed)
    _, _, f_s = run_policy(pick_sm2, True, seed)
    rand_scores.append(f_r.mean() * 100)
    sm2_scores.append(f_s.mean() * 100)

rand_scores, sm2_scores = np.array(rand_scores), np.array(sm2_scores)
diff = sm2_scores - rand_scores

print(f"{REPEATS}回くり返した、最終日の平均")
print(f"  ランダム: {rand_scores.mean():.1f}%  (ばらつき ±{rand_scores.std():.1f})")
print(f"  SM-2    : {sm2_scores.mean():.1f}%  (ばらつき ±{sm2_scores.std():.1f})")
print(f"  差(SM-2 - ランダム): {diff.mean():+.1f} ポイント")
print(f"  SM-2 の方が高かった回数: {REPEATS}回中 {(diff > 0).sum()}回")

plt.figure(figsize=(6, 3.5))
plt.hist(diff, bins=10, color="tab:blue", alpha=0.7)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("差(SM-2 − ランダム)ポイント")
plt.ylabel("回数")
plt.title("20回くり返した結果の差")
plt.tight_layout()
plt.show()

## 7. 1日の出題数を変えると、結果はどうなるか

ここまでは「1日10問」で比べました。出題数を変えると、結果が変わるかを見ます。
(出題数が多いと、すべての問題を何度も見ることになり、出し方の工夫が効きにくくなるはずです。)

In [ ]:
original_per_day = PER_DAY
sweep = []
for k in [2, 3, 5, 10, 20]:
    PER_DAY = k                       # 1日の出題数を変えて、10回ずつくり返す
    r = [run_policy(pick_random, False, s)[2].mean() * 100 for s in range(10)]
    m = [run_policy(pick_sm2, True, s)[2].mean() * 100 for s in range(10)]
    sweep.append((k, np.mean(r), np.mean(m)))
PER_DAY = original_per_day            # 元の設定に戻す

sweep_df = pd.DataFrame(sweep, columns=["1日の出題数", "ランダム(%)", "SM-2(%)"])
sweep_df["差(SM-2 − ランダム)"] = sweep_df["SM-2(%)"] - sweep_df["ランダム(%)"]

plt.figure(figsize=(6, 3.5))
plt.plot(sweep_df["1日の出題数"], sweep_df["ランダム(%)"], marker="o", color="tab:gray", label="ランダム")
plt.plot(sweep_df["1日の出題数"], sweep_df["SM-2(%)"], marker="o", color="tab:blue", label="SM-2")
plt.xlabel("1日の出題数")
plt.ylabel("最終日の正解できる確率(%)")
plt.title("出題数を変えたときの比較")
plt.legend()
plt.tight_layout()
plt.show()

sweep_df.round(1)

## 8. まとめと、次にやること

**このノートの初期設定で、私が動かしたときの結果**

- 1日の出題数が少ないとき(2〜5問)は、SM-2 がランダムをわずかに上回りました。
- 出題数が多いとき(10問以上)は、逆にランダムの方が高くなりました。
- どの場合も、差は大きくありません。2週間あけると、多くの問題が「勘で当たる確率」(20%)に近づくためです。

**どう読むか**

- この結果は、仮の忘却モデルのもとでの比較です。数字(忘れやすさ、伸び方)を変えれば、結果も変わります。
- このモデルには、「何度も復習しすぎても、効果がだんだん小さくなる」という性質が入っていません。
  そのため、出題数が多いと、とにかく回数を重ねる方式が有利に出ています。
- SM-2 の間隔(1日 → 6日 → …)は、全員に同じです。**得意な生徒にも苦手な生徒にも同じ間隔を使うのは、もったいない**はずです。
  ここが、KNOWLY が「生徒ごとに間隔を変える」ことで改善できる余地です。

なので、これは**効果の証明ではなく、「記録 → 分析 → 出題」の流れが動くことの確認**です。
本物の効果は、実際に使ってもらったデータで確かめます。

**次にやること**

- [ ] 作った記録(`data/log_*.csv`)を読み込んで、生徒ごと・問題ごとの傾向を見る(正答率、解答時間)
- [ ] 「正解できる確率」を、記録から推定する方法を作る(いまは仮のモデルの答えを直接使っている)
- [ ] 生徒ごとに間隔を変える方式を作って、ランダム・SM-2 と比べる
- [ ] 本物のデータが取れたら、同じ手順で比べ直す